# Simple RAG over 7 Career PDFs (Google Colab)

**Pipeline:** PDF extraction → cleaning → chunking → `all-MiniLM-L6-v2` embeddings → FAISS → retrieval → cross-encoder reranking → RRF fusion → context → **Groq API (`openai/gpt-oss-120b`)** → answer + sources.

This is the final cleaned-up version of the notebook:

- No local model server (Ollama removed). The LLM call goes to the **Groq API**, which is also used by the companion `app.py` deployed on Streamlit Community Cloud, so the notebook and deployed app use the same overall RAG pipeline and LLM configuration.
- The reranking problem where a relevant chunk could receive an unusually low cross-encoder score and be dropped is handled using **Reciprocal Rank Fusion (RRF)**. RRF combines the rankings from the original retrieval step and the cross-encoder reranking step instead of relying only on the raw cross-encoder scores.
- Debug and duplicate exploration cells have been removed. The notebook is designed to run from top to bottom using **Runtime → Run all**.

**Before running:** upload the 7 PDFs into the Colab file browser in the same folder as the notebook, and have a Groq API key ready.

## 1. Setup

In [1]:
!pip install -q pypdf langchain-text-splitters sentence-transformers faiss-cpu requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.1 MB/s eta 0:00:00


In [3]:
import os, re, json
import numpy as np
import requests
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# ---- Fixed pipeline configuration ----
PDF_DIR = "."  # folder where the 7 PDFs are uploaded in Colab

# Embedding model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Cross-Encoder reranking model
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Chunking configuration
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Retrieval configuration
TOP_K = 8            # number of candidate chunks retrieved by FAISS
TOP_N_RERANK = 4     # number of final chunks kept after reranking + RRF

# ---- LLM (Groq API, OpenAI-compatible endpoint) ----
GROQ_MODEL = "openai/gpt-oss-120b"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

print("Config loaded.")
print(
    f"CHUNK_SIZE={CHUNK_SIZE}  "
    f"CHUNK_OVERLAP={CHUNK_OVERLAP}  "
    f"TOP_K={TOP_K}  "
    f"TOP_N_RERANK={TOP_N_RERANK}"
)

Config loaded.
CHUNK_SIZE=1000  CHUNK_OVERLAP=200  TOP_K=8  TOP_N_RERANK=4


### Groq API key

We never hardcode API keys in the notebook or in `app.py`. Here we ask for the key once at runtime with `getpass` (it is kept only in memory for this Colab session). In the deployed Streamlit app, the same key is read from **Streamlit Secrets** instead (see `app.py` / README).

In [4]:
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (from https://console.groq.com/keys): ")

print("Groq API key is set." if os.environ.get("GROQ_API_KEY") else "WARNING: no API key set.")

Enter your Groq API key (from https://console.groq.com/keys): ··········
Groq API key is set.


## 2. Locate the 7 PDFs

Exact filenames expected in `PDF_DIR`.

In [5]:
EXPECTED_PDFS = [
    "01_Resume_Writing_Best_Practices.pdf",
    "02_How_to_Analyze_a_Job_Description.pdf",
    "03_Behavioral_Interview_Framework.pdf",
    "04_Technical_Interview_and_System_Design.pdf",
    "05_Salary_Negotiation_Playbook.pdf",
    "06_Career_Growth_and_Personal_Branding.pdf",
    "07_Data_Roles_Roadmaps_and_Tools.pdf",
]

all_files = os.listdir(PDF_DIR)
pdf_paths = []
for name in EXPECTED_PDFS:
    path = os.path.join(PDF_DIR, name)
    if os.path.exists(path):
        pdf_paths.append(path)
    else:
        print(f"WARNING: missing {name}")

# Skip any duplicate variant like '...(1).pdf'
duplicates = [f for f in all_files if "(1)" in f and f.lower().endswith(".pdf")]
if duplicates:
    print("Ignoring duplicate file(s):", duplicates)

print(f"PDFs loaded: {len(pdf_paths)}")
for p in pdf_paths:
    print(" -", p)

PDFs loaded: 7
 - ./01_Resume_Writing_Best_Practices.pdf
 - ./02_How_to_Analyze_a_Job_Description.pdf
 - ./03_Behavioral_Interview_Framework.pdf
 - ./04_Technical_Interview_and_System_Design.pdf
 - ./05_Salary_Negotiation_Playbook.pdf
 - ./06_Career_Growth_and_Personal_Branding.pdf
 - ./07_Data_Roles_Roadmaps_and_Tools.pdf


## 3. Extract text (page-by-page, with metadata)

In [6]:
raw_pages = []  # list of dicts: {source, page, text}

for path in pdf_paths:

    filename = os.path.basename(path)

    reader = PdfReader(path)

    n_chars = 0

    for i, page in enumerate(reader.pages):

        text = page.extract_text() or ""

        n_chars += len(text)

        raw_pages.append({
            "source": filename,
            "page": i + 1,
            "text": text
        })

    print(f"{filename}: {len(reader.pages)} pages, {n_chars} characters extracted")

print(f"\nTotal pages extracted across all PDFs: {len(raw_pages)}")

01_Resume_Writing_Best_Practices.pdf: 3 pages, 6479 characters extracted
02_How_to_Analyze_a_Job_Description.pdf: 3 pages, 4565 characters extracted
03_Behavioral_Interview_Framework.pdf: 3 pages, 4391 characters extracted
04_Technical_Interview_and_System_Design.pdf: 3 pages, 5773 characters extracted
05_Salary_Negotiation_Playbook.pdf: 2 pages, 4480 characters extracted
06_Career_Growth_and_Personal_Branding.pdf: 2 pages, 4430 characters extracted
07_Data_Roles_Roadmaps_and_Tools.pdf: 3 pages, 6687 characters extracted

Total pages extracted across all PDFs: 19


## 4. Clean text

In [7]:
def clean_text(text):
    # Normalize bullet characters
    text = re.sub(r"[\u2022\u25a0\u007f]", "-", text)

    # Normalize different types of line breaks
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove spaces/tabs at the beginning and end of lines
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines, but preserve paragraph structure
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove spaces before/after line breaks
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()


cleaned_pages = []

for p in raw_pages:
    ct = clean_text(p["text"])
    if ct:
        cleaned_pages.append({"source": p["source"], "page": p["page"], "text": ct})

print(
    f"Cleaned pages/documents: {len(cleaned_pages)} "
    f"(dropped {len(raw_pages) - len(cleaned_pages)} empty pages)"
)

Cleaned pages/documents: 19 (dropped 0 empty pages)


## 5. Chunking

What is a chunk? A chunk is a small, self-contained slice of a document (roughly `CHUNK_SIZE` characters, with a small overlap between neighboring chunks so context isn't cut mid-idea). We chunk because embedding models and the LLM context window work best on small, focused pieces of text rather than whole documents.

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",  # paragraph / section
        "\n",    # line / bullet
        ". ",    # sentence
        " ",     # word
        ""       # character fallback
    ],
)

chunks = []  # {text, source, page, chunk_id}
chunk_id = 0

for p in cleaned_pages:

    pieces = splitter.split_text(p["text"])

    for piece in pieces:

        chunks.append({
            "text": piece.strip(),
            "source": p["source"],
            "page": p["page"],
            "chunk_id": chunk_id,
        })

        chunk_id += 1

print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")
print(f"Total number of chunks: {len(chunks)}")

print("\nExample chunks with metadata:")

for c in chunks[:3]:

    print("=" * 80)
    print(f"[chunk_id={c['chunk_id']}] source={c['source']} | page={c['page']}")
    print(c["text"])

Chunk size: 1000
Chunk overlap: 200
Total number of chunks: 52

Example chunks with metadata:
[chunk_id=0] source=01_Resume_Writing_Best_Practices.pdf | page=1
Resume Writing Best Practices
CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.
1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who
typically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.
A resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a
person; a beautifully designed two-column resume may impress a human but fail to parse at all.
What actually changed with AI screening
-
Semantic matching, 

## 6. Embeddings

An embedding turns a chunk of text into a vector (a list of numbers) that captures its meaning, so pieces of text with similar meaning end up close together in vector space.

In [9]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in chunks]

embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Number of embeddings: 52
Embedding dimension: 384


## 7. FAISS vector database

FAISS stores all chunk vectors and lets us quickly find the vectors closest (most similar in meaning) to a query vector.

In [10]:
chunk_embeddings = embeddings
embedding_dim = chunk_embeddings.shape[1]

# Normalize embeddings so Inner Product == cosine similarity
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS index size (number of vectors): {index.ntotal}")
print(f"Embedding dimension: {embedding_dim}")

FAISS index size (number of vectors): 52
Embedding dimension: 384


## 8. Retrieval

The retriever embeds the user's query with the *same* embedding model, then asks FAISS for the top-K most similar chunks.

In [11]:
def retrieve(query, k=TOP_K):
    q_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)

    scores, indices = index.search(q_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        c = chunks[idx]
        results.append({**c, "score": float(score)})

    return results


# Quick sanity check on two representative queries
for _q in ["What is the STAR method?", "What are the three tiers of requirements in a job description?"]:
    _demo = retrieve(_q, k=TOP_K)
    print(f"Query: {_q}")
    for r in _demo:
        print(f"  score={r['score']:.3f} | source={r['source']} | page={r['page']} | chunk_id={r['chunk_id']}")
    print()

Query: What is the STAR method?
  score=0.368 | source=03_Behavioral_Interview_Framework.pdf | page=1 | chunk_id=16
  score=0.158 | source=04_Technical_Interview_and_System_Design.pdf | page=1 | chunk_id=25
  score=0.114 | source=03_Behavioral_Interview_Framework.pdf | page=1 | chunk_id=18
  score=0.106 | source=06_Career_Growth_and_Personal_Branding.pdf | page=1 | chunk_id=38
  score=0.088 | source=02_How_to_Analyze_a_Job_Description.pdf | page=2 | chunk_id=13
  score=0.081 | source=07_Data_Roles_Roadmaps_and_Tools.pdf | page=2 | chunk_id=48
  score=0.061 | source=02_How_to_Analyze_a_Job_Description.pdf | page=3 | chunk_id=15
  score=0.057 | source=04_Technical_Interview_and_System_Design.pdf | page=1 | chunk_id=24

Query: What are the three tiers of requirements in a job description?
  score=0.714 | source=02_How_to_Analyze_a_Job_Description.pdf | page=1 | chunk_id=9
  score=0.603 | source=02_How_to_Analyze_a_Job_Description.pdf | page=2 | chunk_id=14
  score=0.539 | source=02_How_to

## 9. Reranking

Retrieval (embeddings + FAISS) is fast but approximate. The reranker is a slower, more precise cross-encoder that directly compares the query and each candidate chunk together, and reorders them by true relevance. We retrieve broadly (Top-K) then rerank down to the best few chunks (Top-N).

### The bug this fixes

For the query *"What are important ATS resume formatting rules?"*, FAISS correctly retrieves the right chunk (`chunk_id=2`, retrieval score ≈ 0.544). But the cross-encoder (`ms-marco-MiniLM-L-6-v2`) happens to score that specific chunk very negatively (≈ -10.5) — a known failure mode of this small cross-encoder on list/formatting-style text. The original notebook combined **raw** retrieval and reranker scores with min-max normalization, and one extreme outlier score was enough to knock the correct chunk out of the Top-N.

### The fix: Reciprocal Rank Fusion (RRF)

Instead of combining raw scores (which live on different, incomparable scales and are sensitive to a single outlier), we combine **ranks**. Each chunk gets a fusion score:

`combined_score = 1 / (RRF_K + retrieval_rank) + 1 / (RRF_K + rerank_rank)`

RRF is a standard, simple, well-known technique in information retrieval (used e.g. in Elasticsearch hybrid search). It is robust to score-scale mismatches and outliers because it only cares about *where* a chunk lands in each ranking, not how extreme the raw score is — a chunk that is retrieval-rank #1 stays competitive even if the cross-encoder scores it oddly on that particular question.

In [18]:
reranker = CrossEncoder(RERANK_MODEL_NAME)

RRF_K = 60  # standard constant used in Reciprocal Rank Fusion


def rerank(query, candidates, top_n=TOP_N_RERANK, rrf_k=RRF_K):

    if not candidates:
        return []

    # Cross-Encoder scores
    pairs = [[query, c["text"]] for c in candidates]

    rerank_scores = reranker.predict(pairs)

    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)

    # Rank by retrieval score (best first)
    by_retrieval = sorted(
        candidates,
        key=lambda c: c["score"],
        reverse=True
    )

    retrieval_rank = {
        c["chunk_id"]: i
        for i, c in enumerate(by_retrieval)
    }

    # Rank by Cross-Encoder score (best first)
    by_rerank = sorted(
        candidates,
        key=lambda c: c["rerank_score"],
        reverse=True
    )

    rerank_rank = {
        c["chunk_id"]: i
        for i, c in enumerate(by_rerank)
    }

    # Reciprocal Rank Fusion
    for c in candidates:

        r1 = retrieval_rank[c["chunk_id"]]
        r2 = rerank_rank[c["chunk_id"]]

        c["combined_score"] = (
            1.0 / (rrf_k + r1 + 1)
            + 1.0 / (rrf_k + r2 + 1)
        )

    # Keep the best final chunks
    reranked = sorted(
        candidates,
        key=lambda c: c["combined_score"],
        reverse=True
    )[:top_n]

    return reranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Verify the fix on the ATS question

In [19]:
query = "What are important ATS resume formatting rules?"

candidates = retrieve(query, k=TOP_K)

reranked = rerank(
    query,
    candidates,
    top_n=TOP_N_RERANK
)

print("Chunk 2 among raw candidates:")

for c in candidates:

    if c["chunk_id"] == 2:

        print(
            f"  retrieval_score={c['score']:.3f} "
            f"source={c['source']} "
            f"page={c['page']}"
        )

print("\nFinal reranked Top-N (RRF):")

kept_ids = []

for r in reranked:

    kept_ids.append(r["chunk_id"])

    print(
        f"  combined_score={r['combined_score']:.4f} | "
        f"rerank_score={r['rerank_score']:.2f} | "
        f"retrieval_score={r['score']:.3f} | "
        f"chunk_id={r['chunk_id']} | "
        f"source={r['source']} | "
        f"page={r['page']}"
    )

print(
    "\nchunk_id=2 kept in final Top-N:",
    2 in kept_ids
)

Chunk 2 among raw candidates:
  retrieval_score=0.544 source=01_Resume_Writing_Best_Practices.pdf page=1

Final reranked Top-N (RRF):
  combined_score=0.0325 | rerank_score=-0.84 | retrieval_score=0.543 | chunk_id=7 | source=01_Resume_Writing_Best_Practices.pdf | page=3
  combined_score=0.0317 | rerank_score=-6.38 | retrieval_score=0.525 | chunk_id=14 | source=02_How_to_Analyze_a_Job_Description.pdf | page=2
  combined_score=0.0315 | rerank_score=-10.53 | retrieval_score=0.544 | chunk_id=2 | source=01_Resume_Writing_Best_Practices.pdf | page=1
  combined_score=0.0315 | rerank_score=-1.58 | retrieval_score=0.505 | chunk_id=0 | source=01_Resume_Writing_Best_Practices.pdf | page=1

chunk_id=2 kept in final Top-N: True


## 10. Context construction

Only the reranked, most relevant chunks are passed to the LLM — never entire PDFs.

In [20]:
def build_context(reranked_chunks):
    parts = []
    for c in reranked_chunks:
        parts.append(f"[Source: {c['source']}, page {c['page']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)

## 11. Generation with Groq (`openai/gpt-oss-120b`)

Ollama has been removed — it required a local model server, which is not available on Streamlit Community Cloud.

**Groq** provides an OpenAI-compatible `/chat/completions` API that is used for the final answer generation in this project.

The notebook and the deployed Streamlit app use the same Groq model:

**`openai/gpt-oss-120b`**

The API key is read from an environment variable in Colab and from Streamlit Secrets in the deployed app. It is never hardcoded.

In [21]:
SYSTEM_PROMPT = (
    "You are a careful assistant that answers ONLY using the provided context. "
    "Do not invent information and do not use outside knowledge. "
    "If the context does not contain enough information to answer, say exactly: "
    "\"I could not find this information in the knowledge base.\" "
    "Be concise and clear. When possible, mention the source filename and page number."
)


def generate_answer(query, context, model=GROQ_MODEL, temperature=0.0, timeout=30):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "ERROR: GROQ_API_KEY is not set. Please provide a Groq API key."

    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using only the context above."

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    try:
        resp = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.RequestException as e:
        return f"ERROR calling Groq API: {e}"

## 12. Complete RAG pipeline

In [22]:
def rag_pipeline(query, k=TOP_K, top_n=TOP_N_RERANK, verbose=True):
    candidates = retrieve(query, k=k)
    reranked = rerank(query, candidates, top_n=top_n)
    context = build_context(reranked)
    answer = generate_answer(query, context)

    sources = sorted({(c["source"], c["page"]) for c in reranked})

    if verbose:
        print(f"Query: {query}\n")
        print(f"Answer:\n{answer}\n")
        print("Sources used:")
        for s, p in sources:
            print(f" - {s}, page {p}")
        print("=" * 80)

    return {"query": query, "answer": answer, "sources": sources, "reranked_chunks": reranked}


result = rag_pipeline("What are important ATS resume formatting rules?")

Query: What are important ATS resume formatting rules?

Answer:
Important ATS‑friendly formatting rules (from **01_Resume_Writing_Best_Practices.pdf**):

* **Single‑column layout** – never use multi‑column designs, tables, or text boxes, because parsers can read them out of order or drop content. (​page 1)  
* **Standard section headers** – use plain headings such as “Experience”, “Education”, “Skills” (no creative labels). (page 1)  
* **Standard fonts** – stick to Arial, Calibri, or Georgia at 10‑12 pt; avoid decorative fonts. (page 1)  
* **File type** – save as a text‑based PDF (not a flattened image/scan) or a .docx if the application requests it. (page 1)  
* **Length** – one page for early‑career candidates; one‑to‑two pages for experienced professionals. (page 1)  
* **No hidden text or keyword‑stuffing** – avoid hidden characters, repeated keywords, or spammy keyword blocks. (page 3)  
* **Keywords** – include target‑job keywords naturally in the summary and bullet points, but

## 13. Testing

Covers each of the 7 PDFs individually plus one cross-document question.

In [23]:
evaluation_questions = [

    "What are the five steps of the system design framework?",

    "What is the STAR method?",

    "What are the three tiers of requirements in a job description?",

    "What should I consider when negotiating salary?",

    "What are the main data roles and their roadmaps?",

    "What are important ATS resume formatting rules?",

    "What are the fundamentals of career growth and personal branding?",

    # Cross-PDF question: needs the job-description tiers
    # AND resume bullet-writing guidance
    "How should I use a job description's requirements to rewrite my resume bullets?",
]

evaluation_results = []

for q in evaluation_questions:

    evaluation_results.append(
        rag_pipeline(q)
    )

Query: What are the five steps of the system design framework?

Answer:
The five‑step system‑design framework is:

1. **Clarify requirements** – define functional needs and non‑functional constraints (scale, latency, consistency vs. availability).  
2. **Estimate scale** – make rough calculations for users, requests‑per‑second, data volume/growth.  
3. **High‑level design** – sketch the major components and data flow before diving into details.  
4. **Deep dive** – explore 1–2 key components in depth (data model, API contract, failure handling).  
5. **Trade‑offs and bottlenecks** – identify what would break at 10× scale and propose changes.  

*Source: 04_Technical_Interview_and_System_Design.pdf, page 2.*

Sources used:
 - 04_Technical_Interview_and_System_Design.pdf, page 2
 - 04_Technical_Interview_and_System_Design.pdf, page 3
 - 07_Data_Roles_Roadmaps_and_Tools.pdf, page 2
 - 07_Data_Roles_Roadmaps_and_Tools.pdf, page 3
Query: What is the STAR method?

Answer:
The STAR method is 